[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C65_ProblemSolving_Communication_Course/05_mock_communication/05_mock_communication.ipynb)

# 05 · 白板沟通与模拟演练（回答结构检查器 / 中英句库 / 10 题模拟引擎 / 24 小时清单）

目标：把「边想边说、结构化表达、卡住怎么办」这些沟通技巧，变成**可以运行、可以量化**的训练工具。
这是本课程的收官 notebook，会把前面四个模块（估算/诊断/权衡/模糊需求澄清）的方法论
统一接入到「一次完整的开放题回答」里。

本 notebook 你会亲手实现：
1. **环境自检**
2. **回答结构检查器** —— 给一段回答文本，自动判断：是否结论先行？是否显式说了假设？是否给了验证方式？
3. **句式库与中英对照表** —— 内置数据结构，按「思考动作/沟通功能」索引
4. **10 题模拟演练引擎**（✏️ 练习）—— 出题 → 计时 → 自评 → 弱项统计
5. **弱项报告生成器**（✏️ 练习）—— 多轮演练后自动指出最该优先练的维度
6. **面试前 24 小时检查清单** —— 结构化 + 中英对照

> 心智模型：**面试官打分的对象不是你脑子里的正确答案，是你嘴里说出来的思考过程。**

## 0 · 环境自检

In [ ]:
import sys
import numpy as np

print('Python :', sys.version.split()[0])
print('numpy  :', np.__version__)
assert sys.version_info >= (3, 8), '需要 Python 3.8+'
assert hasattr(np, 'argsort')
print('\n✅ 环境自检通过：本课不需要 GPU、不需要联网。')

## 1 · 回答结构检查器：三个维度的关键词探测

真实场景下判断「是否结论先行/是否说了假设/是否给了验证方式」需要语义理解，
这里用一个**教学版的关键词+位置探测器**做近似——它足够识别本模块例句里的结构特征，
让你在自己的模拟录音转写文本上也能跑一遍，作为练习时的量化反馈，而不是精确的 NLP 系统。

In [ ]:
CONCLUSION_MARKERS = ['我会', '我建议', '结论是', '我倾向于', '我的判断是', '我先说结论']
ASSUMPTION_MARKERS = ['我假设', '我先假设', '假设是', '如果不对', '按……展开', '我先按']
VERIFY_MARKERS = ['验证', '我会去查', '可以通过', '用……来确认', '对拍', '交叉校验', 'A/B']

def _find_first_position(text, markers):
    """返回文本中最早出现的 marker 的字符位置；都不出现则返回 None。"""
    positions = [text.index(m) for m in markers if m in text]
    return min(positions) if positions else None

def analyze_structure(text, lead_window=40):
    """返回一个 dict：
       has_conclusion / has_assumption / has_verification -> bool
       conclusion_first -> 结论标志是否出现在文本的前 lead_window 个字符内
    """
    c_pos = _find_first_position(text, CONCLUSION_MARKERS)
    a_pos = _find_first_position(text, ASSUMPTION_MARKERS)
    v_pos = _find_first_position(text, VERIFY_MARKERS)
    return {
        'has_conclusion': c_pos is not None,
        'has_assumption': a_pos is not None,
        'has_verification': v_pos is not None,
        'conclusion_first': c_pos is not None and c_pos <= lead_window,
    }

GOOD = ('我建议先做时序投票这一项。理由是：我先假设这次不重训整个检测器，'
        '在这个前提下时序投票成本最低、见效最快。做完之后我会用离线的闪烁率指标验证效果。')
BAD = ('这个问题挺复杂的，可以从很多角度想，比如数据、模型、部署都有可能有问题，'
       '也不太确定具体是哪个，可能要看看情况。')

r_good = analyze_structure(GOOD)
r_bad = analyze_structure(BAD)
assert r_good == {'has_conclusion': True, 'has_assumption': True, 'has_verification': True, 'conclusion_first': True}
assert r_bad == {'has_conclusion': False, 'has_assumption': False, 'has_verification': False, 'conclusion_first': False}

print('好回答的结构诊断:', r_good)
print('差回答的结构诊断:', r_bad)
print('\n✅ 探测器能区分"结论先行+说假设+给验证"与"绕圈子不落地"两种典型回答。')

## 2 · 句式库：按「思考动作/沟通功能」索引（中英对照）

内置一个字典，key 是「沟通功能」，value 是一组中英对照句式。这是配套 notebook 的
「随时可查」版本——正文第 5 节的完整句库以这个数据结构为准。

In [ ]:
PHRASE_BANK = {
    'clarify': [
        ('我先复述一遍，确认我理解对了：……', "Let me restate to make sure I understand correctly: ..."),
        ('您说的这个指标具体指哪个？', "When you say that, which metric are you referring to specifically?"),
    ],
    'assumption': [
        ('我先假设……，如果不对请随时打断我。', "I'll assume ... for now — please stop me if that's not right."),
        ('我先按 A 这个假设往下展开，需要的话再切到 B。', "I'll go with assumption A for now; we can pivot to B if needed."),
    ],
    'tradeoff': [
        ('这里有一个权衡：……', "There's a tradeoff here between ... and ..."),
        ('我选 A 而不是 B，代价是……', "I'm going with A over B; the cost of that is ..."),
    ],
    'uncertainty': [
        ('这一点我不是很确定，我的第一反应是……', "I'm not fully certain here — my first instinct is ..."),
        ('这个数字我不确定，但可以给一个量级估算。', "I don't know the exact number, but I can give an order-of-magnitude estimate."),
    ],
}
FUNCTIONS = list(PHRASE_BANK)
assert FUNCTIONS == ['clarify', 'assumption', 'tradeoff', 'uncertainty']
assert all(len(v) >= 2 for v in PHRASE_BANK.values())

for fn in FUNCTIONS:
    print(f'[{fn}]')
    for cn, en in PHRASE_BANK[fn]:
        print(f'  CN: {cn}')
        print(f'  EN: {en}')
print('\n✅ 句库就位：按功能查，而不是按话题背。')

### 句库实战：把四类功能句拼成一段完整开场白

单独背句子容易，难的是临场把它们串起来。下面拼一段真实会用到的开场白，
再用第 1 节的结构检查器验证它确实拿到了结构满分——这就是句库和检查器该配合使用的方式。

In [ ]:
opening = (
    '我建议先做多帧时序投票这一项。'
    '我先假设这次场景是城市道路，如果不对请随时打断我。'
    '这里有一个权衡：时序投票成本低，但对召回本身提升有限。'
    '这一点我不是很确定，等做完我会用离线的闪烁率指标验证效果。'
)
r_opening = analyze_structure(opening)
assert r_opening['conclusion_first'] is True
assert r_opening['has_assumption'] is True
assert r_opening['has_verification'] is True
manual_score = int(r_opening['conclusion_first']) + int(r_opening['has_assumption']) + int(r_opening['has_verification'])
assert manual_score == 3

print(opening)
print()
print('结构诊断:', r_opening, '  得分:', manual_score, '/3  （这就是练习 1 要实现的 score_answer 逻辑）')
print('\n✅ 句库不是用来单独背的，是用来拼成一段"结论先行+说假设+给验证"的完整开场白。')

## 3 · 10 题模拟演练题库（数据结构）

把正文第 7 节的十道题内置为结构化数据：每题有评分要点 `rubric_points`（3 条）、
满分骨架 `skeleton`（拆成有序步骤）、常见失分点 `pitfall`，以及建议用时 `suggested_min`。

In [ ]:
DRILL_BANK = [
    {'id': 1, 'q': '提升 TSR 效果，你会怎么做？', 'suggested_min': 4,
     'rubric_points': ['是否先收敛边界', '是否给出可执行排序', '是否声明假设并请确认'],
     'skeleton': ['澄清六维度', '收敛出一个假设并声明', 'Impact×Confidence÷Cost 排序', '给分阶段计划'],
     'pitfall': '上来就讲技术方案，没有先定义"效果"是什么'},
    {'id': 2, 'q': '检测器上线一周后指标掉了，你怎么排查？', 'suggested_min': 4,
     'rubric_points': ['是否结构化诊断', '假设是否可证伪', '是否分层定位'],
     'skeleton': ['分清数据/模型/部署哪一层', '按二分法或信息增益排查顺序', '把假设写成可验证的形式'],
     'pitfall': '一次性列出十个可能原因但没有排查顺序'},
    {'id': 3, 'q': '两个月一个工程师，先做精度还是先做部署稳定性？', 'suggested_min': 3,
     'rubric_points': ['是否列维度', '是否显式打分', '是否说清放弃了什么'],
     'skeleton': ['列出比较维度', '打分', '说明帕累托关系与取舍'],
     'pitfall': '只给结论不给理由，或含糊说都重要'},
    {'id': 4, 'q': '只能加一种数据，你会加哪种？', 'suggested_min': 3,
     'rubric_points': ['候选集合是否完整', '是否用打分而非直觉', '排序理由是否可复算'],
     'skeleton': ['列候选方向', 'RICE 打分', '排序并说明理由'],
     'pitfall': '凭直觉说一个方向，没有对比其他候选'},
    {'id': 5, 'q': '估算采集训练数据需要多久。', 'suggested_min': 4,
     'rubric_points': ['是否分解到子任务', '是否用锚点数字', '是否双路径校验'],
     'skeleton': ['分解任务', '给锚点数字', '相乘估算', '换一条路径交叉校验'],
     'pitfall': '直接编一个数字，说不出是怎么算出来的'},
    {'id': 6, 'q': '向非技术产品经理解释方案风险。', 'suggested_min': 3,
     'rubric_points': ['是否结论先行', '是否避免行话', '类比是否贴切'],
     'skeleton': ['先说结论性风险', '用类比替代行话', '给出应对措施'],
     'pitfall': '满口专业术语，PM 听不懂也不敢打断'},
    {'id': 7, 'q': '方案在夜间场景可能失效，怎么和团队沟通？', 'suggested_min': 3,
     'rubric_points': ['是否主动暴露不确定性', '是否给出验证计划', '语气是否诚实'],
     'skeleton': ['说现象', '给出假设根因', '提出验证计划', '说明当前把握程度'],
     'pitfall': '把话说得比实际更有把握，掩盖未验证的部分'},
    {'id': 8, 'q': '预算临时砍半，怎么办？', 'suggested_min': 3,
     'rubric_points': ['是否快速重新排序', '是否说清砍掉了什么', '态度是否积极'],
     'skeleton': ['重算约束下的可行集', '更新排序', '说明砍掉的部分与影响'],
     'pitfall': '抱怨约束不合理，而不是给出调整后的方案'},
    {'id': 9, 'q': '和同事对技术方案有分歧，怎么解决？', 'suggested_min': 3,
     'rubric_points': ['是否给出可验证判据', '是否避免诉诸权威/情绪', '方案是否可执行'],
     'skeleton': ['列出分歧背后的假设', '设计一个能分辨对错的小实验', '按结果决定采用哪个'],
     'pitfall': '说"我经验更多"而不给可验证判据'},
    {'id': 10, 'q': '方案没达到预期，怎么复盘？', 'suggested_min': 3,
     'rubric_points': ['是否承认失败', '归因是否具体', '是否给出下一步行动'],
     'skeleton': ['对比预期与实际', '按数据/模型/评测/沟通归因', '给出下一步行动'],
     'pitfall': '把失败归咎于外部因素，不做自我审视'},
]
assert len(DRILL_BANK) == 10
assert all(len(d['rubric_points']) == 3 for d in DRILL_BANK)
assert all(len(d['skeleton']) >= 3 for d in DRILL_BANK)
assert [d['id'] for d in DRILL_BANK] == list(range(1, 11))
print(f'题库就位：共 {len(DRILL_BANK)} 题，建议总用时 {sum(d["suggested_min"] for d in DRILL_BANK)} 分钟。')

## ✏️ 练习 1：回答结构评分函数

实现 `score_answer(text)`，基于第 1 节的 `analyze_structure`，返回 0-3 的整数分：
`conclusion_first` + `has_assumption` + `has_verification` 各算 1 分（`has_conclusion`
不单独计分——如果结论没出现在前段，只出现在后面，不给结论分，因为这不算"结论先行"）。

In [ ]:
def score_answer(text):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert score_answer(GOOD) == 3, score_answer(GOOD)
assert score_answer(BAD) == 0, score_answer(BAD)

# 结论出现了，但不在开头 —— 不算"结论先行"，这一分拿不到
LATE_CONCLUSION = ('这个问题背景比较复杂，有很多因素需要考虑，' + '占位' * 20 +
                   '综合下来我建议先做时序投票。我先假设不重训模型，做完会用离线指标验证。')
r = score_answer(LATE_CONCLUSION)
assert r == 2, r   # 有假设 + 有验证，但结论不在前 40 字 -> 少 1 分

# 只说了假设，没有验证也没有结论
ONLY_ASSUMPTION = '我先假设这次场景是城市道路。'
assert score_answer(ONLY_ASSUMPTION) == 1, score_answer(ONLY_ASSUMPTION)

for label, t in [('好回答', GOOD), ('差回答', BAD), ('结论靠后', LATE_CONCLUSION), ('只有假设', ONLY_ASSUMPTION)]:
    print(f'{label:<8} -> {score_answer(t)}/3')
print('\n✅ 练习 1 通过：结论先行不是"有没有结论"，是"结论出没出现在开头"。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def score_answer(text):
    r = analyze_structure(text)
    return int(r['conclusion_first']) + int(r['has_assumption']) + int(r['has_verification'])

## 4 · 模拟演练引擎（worked）：出题 → 计时 → 自评

`run_drill(drill_id, elapsed_sec, self_scores)` 模拟一次演练的记录动作：
给定题号、实际用时（秒）与自评分（三个维度各 0/1），返回一条结构化记录，
供后续弱项统计使用。这里先给出 worked 版本，练习 2 会在此基础上实现多轮统计。

In [ ]:
DRILL_BY_ID = {d['id']: d for d in DRILL_BANK}

def run_drill(drill_id, elapsed_sec, self_scores):
    """self_scores: dict，如 {'clarity': 1, 'assumption': 1, 'verification': 0}"""
    d = DRILL_BY_ID[drill_id]
    over_time = elapsed_sec > d['suggested_min'] * 60
    return {
        'id': drill_id,
        'q': d['q'],
        'elapsed_sec': elapsed_sec,
        'suggested_sec': d['suggested_min'] * 60,
        'over_time': over_time,
        'self_scores': dict(self_scores),
        'total': sum(self_scores.values()),
    }

rec = run_drill(1, elapsed_sec=200, self_scores={'clarity': 1, 'assumption': 1, 'verification': 0})
assert rec['over_time'] is False   # 200s < 4*60=240s
assert rec['total'] == 2
rec2 = run_drill(1, elapsed_sec=300, self_scores={'clarity': 1, 'assumption': 1, 'verification': 1})
assert rec2['over_time'] is True   # 300s > 240s

print(rec)
print(rec2)
print('\n✅ 单次演练记录就位，练习 2 要把多轮记录汇总成弱项报告。')

## ✏️ 练习 2：弱项统计与报告生成器

实现 `weakness_report(records)`：输入是若干条 `run_drill` 返回的记录（列表），
返回一个 dict：
- `avg_by_dim`：三个维度（`clarity`/`assumption`/`verification`）各自的平均自评分（浮点数）
- `weakest_dim`：平均分最低的维度名（并列时取字典序最小的那个）
- `over_time_rate`：`over_time=True` 的记录占比

In [ ]:
def weakness_report(records):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
RECORDS = [
    run_drill(1, 200, {'clarity': 1, 'assumption': 1, 'verification': 0}),
    run_drill(2, 200, {'clarity': 1, 'assumption': 0, 'verification': 0}),
    run_drill(3, 150, {'clarity': 1, 'assumption': 1, 'verification': 1}),
    run_drill(4, 400, {'clarity': 0, 'assumption': 0, 'verification': 0}),
]
rep = weakness_report(RECORDS)

assert abs(rep['avg_by_dim']['clarity'] - 0.75) < 1e-9        # (1+1+1+0)/4
assert abs(rep['avg_by_dim']['assumption'] - 0.5) < 1e-9        # (1+0+1+0)/4
assert abs(rep['avg_by_dim']['verification'] - 0.25) < 1e-9     # (0+0+1+0)/4
assert rep['weakest_dim'] == 'verification'
assert abs(rep['over_time_rate'] - 0.25) < 1e-9                 # 只有题 4 超时(400s > 3*60=180s)

for dim, avg in rep['avg_by_dim'].items():
    print(f'{dim:<12} 平均分 {avg:.2f}')
print(f"最弱维度: {rep['weakest_dim']}   超时率: {rep['over_time_rate']:.0%}")
print('\n✅ 练习 2 通过：verification（给验证方式）是这组演练里最该优先练的维度。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 2 参考答案
def weakness_report(records):
    dims = ['clarity', 'assumption', 'verification']
    avg_by_dim = {dim: sum(r['self_scores'][dim] for r in records) / len(records) for dim in dims}
    weakest_dim = min(dims, key=lambda d: (avg_by_dim[d], d))
    over_time_rate = sum(1 for r in records if r['over_time']) / len(records)
    return {'avg_by_dim': avg_by_dim, 'weakest_dim': weakest_dim, 'over_time_rate': over_time_rate}

## ✏️ 练习 3：卡住检测与脱困提示器

实现 `stuck_advice(silence_sec)`，根据沉默时长返回建议采取的脱困法（对应正文第 3 节）：
- `silence_sec < 10` → `'ok'`（还不到需要动作的程度）
- `10 <= silence_sec < 20` → `'step_back'`（建议：退回上一层）
- `20 <= silence_sec < 30` → `'concrete_example'`（建议：举具体例子）
- `silence_sec >= 30` → `'ask_for_time'`（建议：明说需要一分钟——**这是最后的底线，
  绝不能选择继续沉默**）

In [ ]:
def stuck_advice(silence_sec):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert stuck_advice(5) == 'ok'
assert stuck_advice(10) == 'step_back'
assert stuck_advice(19) == 'step_back'
assert stuck_advice(20) == 'concrete_example'
assert stuck_advice(29) == 'concrete_example'
assert stuck_advice(30) == 'ask_for_time'
assert stuck_advice(120) == 'ask_for_time'   # 沉默 2 分钟也不会有第五种建议，永远收敛到"明说要时间"

for s in (5, 12, 22, 35, 60):
    print(f'沉默 {s:>3}s -> {stuck_advice(s)}')
print('\n✅ 练习 3 通过：无论沉默多久，系统永远给出一个"该说什么"，而不是"继续等"。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 3 参考答案
def stuck_advice(silence_sec):
    if silence_sec < 10:
        return 'ok'
    if silence_sec < 20:
        return 'step_back'
    if silence_sec < 30:
        return 'concrete_example'
    return 'ask_for_time'

---
## 🧪 真实工程胶囊：面试前 24 小时检查清单（中英对照）

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# A. 面试前 24 小时检查清单
# ══════════════════════════════════════════════════════════════════════
# □ 形式确认：白板 / 共享屏幕 / 纯语音，提前用同款工具跑一遍（尤其是没有
#             语法高亮和自动缩进的协作文档，务必先适应摩擦）
# □ 句库过一遍：clarify / assumption / tradeoff / uncertainty 四类中英各背 2 句，
#             练到"动作触发句子"而不是"回忆句子"
# □ 十题过一遍骨架（不是背答案）：每题只回忆 skeleton 的 3-4 步，
#             忘了哪步就回去看正文表格
# □ 收尾话术准备好一句可复用的模板：
#             "我先总结一下：方案是……，关键假设是……，最大风险是……，
#              如果有更多时间我会验证……"
# □ 反问准备 1-2 个真实问题（不要问网上查得到的）
# □ 心态：目标不是"讲得完美"，是"讲得让人看见你在想什么"

# ══════════════════════════════════════════════════════════════════════
# B. 四类关键英文表达速查（各挑最常用的一句背下来）
# ══════════════════════════════════════════════════════════════════════
# 澄清:   "Let me restate to make sure I understand correctly: ..."
# 假设:   "I'll assume ... for now — please stop me if that's not right."
# 权衡:   "There's a tradeoff here between ... and ...; I'm going with ... because ..."
# 不确定: "I'm not fully certain here — my first instinct is ..., but I'd want to double-check."

# ══════════════════════════════════════════════════════════════════════
# C. 卡住时的行动表（沉默超过 30 秒是硬扣分线）
# ══════════════════════════════════════════════════════════════════════
# < 10s   继续想，无需动作
# 10-20s  退回上一层："我退一步——最初要解决的是……"
# 20-30s  举具体例子："我拿一个具体例子走一遍：……"
# >= 30s  明说要时间："给我 20-30 秒，我在权衡……"（绝不能选择继续沉默）

# ══════════════════════════════════════════════════════════════════════
# D. 与本课程其他部分的分工（收官提醒）
# ══════════════════════════════════════════════════════════════════════
# · 澄清问题清单与收敛话术         -> C65-04（本模块第7节10题里反复复用）
# · 估算/诊断/权衡的具体方法论     -> C65-01 / C65-02 / C65-03
# · 项目叙事（讲过去做过的事）     -> C61-04（对象不同：过去 vs 当场开放题）
# · ML 系统设计的完整案例演练脚本  -> C63-05（同一种训练形式，题型更大）
'''
print(RECIPE)
for token in ['24 小时', 'restate', "I'll assume", '30 秒', 'C65-04', 'C63-05']:
    assert token in RECIPE, token
print('✅ 检查单覆盖：赛前清单 / 中英速查 / 卡壳行动表 / 课程收官分工')

### 小结

- **面试官打分的对象是你说出来的思考过程，不是你脑子里的正确答案。** 边想边说需要一套
  「思考动作 → 触发句式」的固化映射，而不是临场现想怎么表达。
- **结构化表达的核心工具是结论先行 / 金字塔原理 / 三点法**——先给答案，
  再给不超过三条的支撑论据，让面试官可以随时决定是否要下钻细节。
- **卡住时永远从三种脱困法里选一个说出口**（退回上一层 / 举具体例子 / 明说要时间），
  沉默本身才是失分项，卡壳不是。
- **三种沟通形式（白板/共享屏幕/纯语音）对结构化表达的要求依次递增**，
  纯语音下必须靠语言本身显式建立层次，不能依赖画面兜底。
- **英文的「让步句式」（I'll assume ... but / I'm not certain, but my instinct is ...）
  需要被专门练习**，直译成生硬的 "I don't know" 会传递错误的信号——一定要在后面接一句你的判断或验证计划。
- **10 道开放题看起来主题各异，骨架高度重复**：定义边界 → 结构化展开 → 显式权衡/假设 →
  主动暴露不确定性 → 给出下一步。这正是 C65-00 四步框架在不同外壳下的反复出现。

到此，C65 全课收官：从估算（01）、诊断（02）、权衡（03）、模糊需求澄清（04），
到本模块的白板沟通与模拟演练（05）——五个模块共同训练的是同一件事：
**让面试官清楚看见你的思考过程，而不只是看见你的结论。**